In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"


c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2.0 — Instruction Fine-Tuning on the Domain-Adapted Model

## Purpose
This notebook implements the **second stage** of the BPMN language model training pipeline. It takes the domain-adapted model from Notebook 1.0 and fine-tunes it further to follow structured **instruction prompts**.

## What This Notebook Does
1. **Loads** the LoRA adapter checkpoint from Notebook 1.0 and merges it into the base TinyLlama weights to produce a clean base for instruction tuning.
2. **Tests** the merged non-instruction model as a baseline to see what domain knowledge has already been learned.
3. **Formats** a curated BPMN instruction dataset into the `### Instruction / ### Input / ### Response` prompt template.
4. **Applies response-only masking** so the model learns to generate answers only, not the instruction prefix.
5. **Fine-tunes** with a new, wider LoRA adapter (rank 16, targeting all q/k/v/o attention projections).
6. **Evaluates and compares** outputs from the non-instruction model vs. the instruction-tuned model.

## Input
- `./tinyllama-lora/checkpoint-370` — LoRA adapter from Notebook 1.0.
- `data/bpmn_instruction_dataset.jsonl` — curated BPMN Q&A pairs in instruction format.

## Output
- `./tinyllama-instruction/` — the instruction-tuned LoRA adapter, used as input for DPO alignment in Notebook 3.0.

---

## Step 1 — Import Libraries

The cell above imports all required libraries for this notebook:
- **`torch`** — PyTorch, the deep learning backend.
- **`transformers`** — Hugging Face library for model loading and training.
- **`peft`** — Parameter-Efficient Fine-Tuning (LoRA adapters).
- **`datasets`** — Hugging Face dataset utilities for loading the instruction data.

In [ ]:

model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
model_path = "./tinyllama-lora/checkpoint-370"

## Step 2 — Define Model and Checkpoint Paths

Set the Hugging Face Hub model ID for the base model and the path to the LoRA adapter checkpoint saved by Notebook 1.0.

> **Note:** `checkpoint-370` is the final checkpoint produced by the 5-epoch domain-adaptation run in Notebook 1.0. If you retrain with a different number of steps, update this path accordingly.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float32, low_cpu_mem_usage=True)
non_instruction_model = PeftModel.from_pretrained(base_model, model_path)
non_instruction_model = non_instruction_model.merge_and_unload()

# PEFT stores state inside layer modules even after merge_and_unload(),
# so del peft_config is not enough. Save + reload as plain weights — the only
# guaranteed way to get a clean model with no PEFT internals.
MERGED_PATH = "./tinyllama-merged-temp"
non_instruction_model.save_pretrained(MERGED_PATH)
non_instruction_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
non_instruction_model = non_instruction_model.to(device)
print("Loaded clean merged model, type:", type(non_instruction_model).__name__)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2029.70it/s]

Loaded clean merged model, type: LlamaForCausalLM


## Step 3 — Baseline Test: Non-Instruction Model Output

Before instruction tuning, run a quick generation test to observe how the non-instruction (domain-adapted) model responds to a BPMN-related prompt.

The prompt `"The Loop Activity is a type of Activity that "` is an open-text continuation — there is **no instruction format**. The model should complete it using BPMN domain knowledge acquired in Notebook 1.0, but without structured question-answering ability.

## Step 4 — Load the Instruction Training Dataset

Load two representative examples from `bpmn_instruction_dataset.jsonl`:
- **Index 0** — A basic BPMN definition question.
- **Index 16** — A question about all five BPMN 2.0 Gateway types.

Using only two examples is intentional for this proof-of-concept: the goal is to verify that the model can memorise and recall structured instruction-response pairs. Generalisation to unseen questions requires a larger dataset and more training steps.

In [5]:

prompt = "The Loop Activity is a type of Activity that "

inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

The Loop Activity is a type of Activity that 251 Business Process Model and Notation, v2.0 A Loop Activity can be either a Recurrence Loop or a Repetition Loop. A Recurrence Loop represents a sequence where each occurrence (i.e., each time) of the activity will run. A Repetition Loop represents an infinite repetition of the activity, i.e., the activity runs continuously. A Loop Activity does not have any inherent timing constraints. For more information on how to define the behavior


In [6]:
dataset = load_dataset("json", data_files="data/bpmn_instruction_dataset.jsonl", split="train")
dataset = dataset.select([0, 16])   # 2 examples: basic BPMN + gateway
print("Training questions:")
for ex in dataset:
    print(" -", ex["instruction"])


Training questions:
 - What is BPMN and what is its primary goal?
 - Describe all five Gateway types in BPMN 2.0 and when to use each.


## Step 5 — Preview a Dataset Example

Inspect the raw instruction text from the first training example to confirm the dataset loaded correctly and to understand the format of the instruction field before formatting.

In [7]:
dataset['instruction'][0]

'What is BPMN and what is its primary goal?'

## Step 6 — Format Examples into the Instruction Template

The `format_example` function converts each raw `{instruction, input, output}` record into a single string using the **Alpaca-style prompt format**:

```
### Instruction:
<instruction text>
### Input:
<optional input context>
### Response:
<expected output>
```

This formatting is critical because the model must learn to associate the `### Response:` marker with the start of the answer. The full formatted string (including instruction + response) is stored in the `"text"` field for subsequent tokenization.

In [8]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

## Step 7 — Ensure Pad Token Is Set

Since TinyLlama's tokenizer does not define a `pad_token` by default, we set it to `eos_token`. This is required before tokenization to avoid errors when padding sequences to a fixed length. (This guard is checked again here because the tokenizer may have been re-instantiated in the session.)

In [9]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Step 8 — Tokenize with Response-Only Label Masking

The `tokenize_and_mask` function is the core of instruction fine-tuning. It tokenizes the full formatted text but **masks the loss on the instruction prefix**, so the model is only trained to predict the response tokens.

**Why masking matters:**
- Without masking, the model is penalised for not perfectly reproducing the instruction text it was given as input — which wastes training signal.
- With masking, all token positions before `### Response:\n` receive label `-100` (ignored by the cross-entropy loss), and all padding tokens beyond the actual sequence end are also masked.

**Special care for `pad_token == eos_token`:**
Because TinyLlama uses `eos_token` as the padding token, trailing padding positions must be explicitly masked. Otherwise the model would be trained to predict `EOS` at every padding position, causing it to generate blank (empty) outputs at inference time.

In [10]:
def tokenize_and_mask(example):
    text = example["text"]

    # Include the newline after ### Response: so the template itself is masked
    response_marker = "### Response:\n"
    response_start = text.find(response_marker)

    # Tokenize full sequence with padding
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = enc["input_ids"]

    if response_start != -1:
        prompt_prefix = text[:response_start + len(response_marker)]
        response_token_start = len(tokenizer(prompt_prefix, truncation=True, max_length=512)["input_ids"])
    else:
        response_token_start = 0

    # Actual (unpadded) sequence length — used to mask trailing padding
    # CRITICAL: pad_token == eos_token, so we MUST NOT label trailing pad positions
    # as valid targets or the model trains to predict EOS everywhere → blank output
    actual_len = len(tokenizer(text, truncation=True, max_length=512)["input_ids"])

    labels = input_ids.copy()
    # Mask prompt + ### Response:\n prefix
    for i in range(min(response_token_start, 512)):
        labels[i] = -100
    # Mask trailing padding (positions beyond actual sequence end)
    for i in range(actual_len, 512):
        labels[i] = -100

    enc["labels"] = labels
    return enc


## Step 9 — Apply Instruction Formatting to the Dataset

Map the `format_example` function over the dataset to convert each `{instruction, input, output}` record into a single `"text"` string using the `### Instruction / ### Response` template. The output is inspected to confirm the formatting is correct before tokenization.

In [11]:
dataset = dataset.map(format_example)
dataset[0]

Map: 100%|██████████| 2/2 [00:00<00:00, 40.64 examples/s]


{'instruction': 'What is BPMN and what is its primary goal?',
 'input': '',
 'output': 'BPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. BPMN creates a standardized bridge between business process design and process implementation. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation.',
 'text': '### Instruction:\nWhat is BPMN and what is its primary goal?\n### Input:\n\n### Response:\nBPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business

## Step 10 — Apply Tokenization and Label Masking

Apply `tokenize_and_mask` to the formatted dataset. Each example is processed individually (`batched=False`) because the masking logic requires per-example string offset calculation. After this step, the dataset contains `input_ids`, `attention_mask`, and `labels` columns ready for training.

In [12]:
# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

Map: 100%|██████████| 2/2 [00:00<00:00, 35.20 examples/s]

Tokenization + masking done.


## Step 11 — Verify Label Masking is Correct

Before training, sanity-check the tokenized labels:
- **Prompt masked** — the number of leading `-100` positions (instruction prefix, including `### Response:\n`).
- **Response tokens** — positions with valid (non–`-100`) labels; these are what the model trains to predict.
- **Trailing masked** — padding positions correctly excluded from the loss.

This verification ensures the masking logic works and avoids silent bugs that could degrade model performance (e.g. training the model to predict padding tokens).

In [13]:
# Verify labels are correct before training
sample = tokenized[0]
labels = sample["labels"]
valid = [i for i, l in enumerate(labels) if l != -100]
print(f"Total tokens: 512 | Prompt masked: {valid[0]} tokens | Response tokens with valid labels: {len(valid)} | Trailing masked: {512 - valid[-1] - 1}")
print("First 5 response token labels (should be real token IDs, not -100):", labels[valid[0]:valid[0]+5])
print("Decoded response (first 80 chars):", tokenizer.decode([l for l in labels if l != -100])[:80])


Total tokens: 512 | Prompt masked: 30 tokens | Response tokens with valid labels: 140 | Trailing masked: 342
First 5 response token labels (should be real token IDs, not -100): [29933, 13427, 29940, 15028, 363]
Decoded response (first 80 chars): BPMN stands for Business Process Model and Notation. It is an OMG standard (vers


## Step 12 — Configure LoRA for Instruction Fine-Tuning

A **wider LoRA adapter** is used for instruction tuning compared to the domain-adaptation stage (Notebook 1.0):

| Parameter | Notebook 1.0 | Notebook 2.0 | Reason for Change |
|-----------|-------------|-------------|-------------------|
| `r` | 8 | **16** | Higher rank for more expressive instruction-following capacity. |
| `lora_alpha` | 16 | **32** | Kept at `2 × r` to maintain the same effective step size. |
| `target_modules` | q, v | **q, k, v, o** | Adding key and output projections doubles trainable params, improving task alignment. |

The larger adapter is justified because instruction following is a more complex behaviour than passive domain adaptation.

In [14]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # was 8 — more rank = more expressiveness per step
    lora_alpha=32,               # keep alpha = 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # was only q+v; adding k+o doubles trainable params
    bias="none",
)


## Step 13 — Apply LoRA to the Merged Model and Print Trainable Parameters

Apply the LoRA configuration to the merged non-instruction model. `print_trainable_parameters()` confirms that:
- Only the LoRA adapter parameters are set as trainable (roughly 4.5 M parameters out of ~1.1 B total — less than 0.5%).
- No "second time" PEFT warning appears, confirming the base model is clean (no leftover PEFT internals from the merge operation).

In [15]:
instruction_model = get_peft_model(non_instruction_model, lora_config)
instruction_model.print_trainable_parameters()
# Should show ~4.5M trainable params and NO "second time" warning


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## Step 14 — Set Training Arguments for Instruction Tuning

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `num_train_epochs` | 50 | 50 × 2 examples ÷ batch size 1 = 100 gradient steps. Enough to memorise the two training examples. |
| `per_device_train_batch_size` | 1 | One example at a time due to small dataset size. |
| `gradient_accumulation_steps` | 1 | No accumulation needed with a batch size of 1. |
| `learning_rate` | 5e-4 | Slightly higher than stage 1 to drive faster convergence on the small instruction set. |
| `fp16` | GPU-conditional | Mixed precision only when a GPU is available; CPU training uses float32. |

In [16]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=50,             # 50 × 2 examples ÷ batch 1 = 100 steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
)


## Step 15 — Initialise the Instruction Trainer

Create the Hugging Face `Trainer` for instruction fine-tuning, passing the LoRA-wrapped instruction model, training arguments, and the tokenized (masked) dataset. No evaluation dataset is provided — correctness is assessed via manual generation tests after training.

In [17]:
trainer = Trainer(
    model=instruction_model     ,
    args=args,
    train_dataset=tokenized,
)

## Step 16 — Train the Instruction Model

Run the instruction fine-tuning loop. The model learns to generate structured responses using the `### Response:` template, leveraging the BPMN domain knowledge already embedded by the domain-adaptation stage.

> **Expected outcome:** Training loss should decrease steadily. With 50 epochs and 2 examples, the model should be able to recall both training questions accurately. Generalisation to new questions is limited without a larger dataset, but the pipeline structure is validated.

In [18]:
trainer.train()

c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,0.684883
40,0.007259
60,0.000905
80,0.000544
100,0.000458


TrainOutput(global_step=100, training_loss=0.13880993902683259, metrics={'train_runtime': 1895.6601, 'train_samples_per_second': 0.053, 'train_steps_per_second': 0.053, 'total_flos': 319186324684800.0, 'train_loss': 0.13880993902683259, 'epoch': 50.0})

## Step 17 — Save the Instruction-Tuned Model

Save the LoRA adapter weights and tokenizer configuration to `./tinyllama-instruction/`.

> **Note on `safe_serialization=False`:** On Windows, PyTorch's `safetensors` serialisation can fail with an OS memory-mapping error (Error 1224) when saving large files. Using the standard pickle-based format avoids this issue.

In [20]:
# Save the model — safe_serialization=False avoids the Windows mmap conflict (os error 1224)
instruction_model.save_pretrained("./tinyllama-instruction", safe_serialization=False)
tokenizer.save_pretrained("./tinyllama-instruction")


('./tinyllama-instruction\\tokenizer_config.json',
 './tinyllama-instruction\\tokenizer.json')

## Step 18 — Test the Instruction-Tuned Model

Run a generation test using a question from the training set (`index 16` — the five BPMN Gateway types). Greedy decoding (`do_sample=False`) is used here because it produces deterministic, reproducible output and avoids randomly sampling `EOS` on a weakly-trained model.

**What to look for:** The model should generate a well-structured answer describing the five BPMN 2.0 Gateway types. If the output is blank or incoherent, it indicates a label-masking bug or insufficient training.

In [22]:
instruction_model.eval()

# Use a question that IS in the training data (idx 16).
# With only ~15 optimizer steps the model can only recall training examples — it
# cannot generalize to paraphrases like "What is gateway?" yet.
prompt = "### Instruction:\nDescribe all five Gateway types in BPMN 2.0 and when to use each.\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,          # greedy — avoids randomly sampling EOS on a weakly-trained model
        repetition_penalty=1.2,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print("\nModel Output:\n")
print(tokenizer.decode(new_tokens, skip_special_tokens=True))



Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

BPMN 2.0 defines five Gateway types, all depicted as diamonds:

1. Exclusive Gateway (XOR) – Only one outgoing path is taken based on conditions. At merge, the first arriving token passes through immediately. Icon: X inside diamond. Use when exactly one of several paths should execute.

2. Inclusive Gateway (OR) – One or more outgoing paths are taken based on conditions. At merge, waits for all active incoming tokens to arrive before continuing. Icon: circle inside diamond. Use when any combination of paths may be active.

3. Parallel Gateway (AND) – All outgoing paths are always taken (fork). At merge (join), waits for all incoming paths to arrive before proceeding. Icon: + inside diamond. Use for unconditional parallel execution.

4. Event-Based Gateway – Routes


## Step 19 — Define Comparison Questions

Define the two questions that were used in instruction training. These will be posed to both the non-instruction model and the instruction-tuned model in the next cell to produce a side-by-side comparison of their outputs.

In [23]:
questions = [
    "What is BPMN and what is its primary goal?",                         # idx 0 — in training
    "Describe all five Gateway types in BPMN 2.0 and when to use each.", # idx 16 — in training
]


## Step 20 — Side-by-Side Model Comparison

For each question, generate output from both models:
- **Non-instruction model** — given the question as plain text, without any template formatting.
- **Instruction-tuned model** — given the question wrapped in the `### Instruction / ### Response` template.

This comparison highlights the qualitative improvement in structured, task-aligned responses gained through instruction fine-tuning. The non-instruction model will likely generate free-form BPMN text, while the instruction model should produce a direct, well-structured answer.

> **Next step:** The instruction-tuned model is further refined using **DPO (Direct Preference Optimisation)** in *Notebook 3.0* to align its outputs with human preferences.

In [ ]:
for q in questions:
    print("Question:", q)

    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to(device)
    outputs = non_instruction_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.3,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.3,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("=" * 80, "\n")


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is BPMN and what is its primary goal?

--- Non-instruction model ---


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A. The Business Process Model and Notation (BPMN), released in 2013, is a standard for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. Bpm

--- Instruction-tuned model ---


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. BPMN creates a standardized bridge between business process design and process implementation. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation. A final but lesser goal is to establish a

Question: Describe all five Gateway types in BPMN 2.0 and when to use each.

--- Non-instruction model ---


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A Gateway is a execution path that waits for conditions to be met before continuing, or activates behavior based on those conditions. Each type of Gateway is identified by a icon in the XML graph representation (Figure 13.4). At first blush, Bridge appears to be the most complex Gateword ever created—with 65 different patterns! In fact, it

--- Instruction-tuned model ---
BPMN 2.0 defines five Gateway types, all depicted as diamonds:

1. Exclusive Gateway (XOR) – Only one outgoing path is taken based on conditions. At merge, the first arriving token passes through immediately. Icon: X inside diamond. Use when exactly one of several paths should execute.

2. Inclusive Gateway (OR) – One or more outgoing paths are taken based on conditions. At merge, waits for all active incoming tokens to arrive before continuing. Icon: circle inside diamond. Use when any combination of paths may be active.

3. Parallel Gateway (AND) – All outgoing paths are always



: 